# MCE Temporal Features: Maya Calendar Encoding

This notebook explores the MayaCalendarEncoder for temporal feature engineering,
demonstrating how Maya calendar cycles can enrich time series data.

In [ ]:
import numpy as np
from maya_encoding import MayaCalendarEncoder
from maya_encoding.core.calendar import (
    gregorian_to_jdn, jdn_to_tzolkin, jdn_to_haab,
    jdn_to_long_count, is_wayeb,
    TZOLKIN_DAY_NAMES, HAAB_MONTH_NAMES,
)

## 1. The Three Maya Calendar Systems

### Tzolk'in (Sacred Calendar)
- 260-day cycle = 13 numbers × 20 day names
- The two cycles (13 and 20) are coprime, creating 260 unique combinations

### Haab' (Solar Calendar)
- 365-day cycle = 18 months × 20 days + 5 "Wayeb'" days
- Each month has 20 days (0-19), plus 5 unlucky days

### Long Count
- Linear count from Maya epoch (August 11, 3114 BCE)
- Mixed radix: 20 (kin), 18 (uinal), 20 (tun), 20 (k'atun), 20 (b'ak'tun)

In [ ]:
# Explore notable dates
dates = [
    (2012, 12, 21, "End of 13th b'ak'tun"),
    (2024, 1, 1, "New Year 2024"),
    (2024, 3, 20, "Spring Equinox 2024"),
    (2024, 6, 21, "Summer Solstice 2024"),
    (2024, 12, 21, "Winter Solstice 2024"),
    (2000, 1, 1, "Y2K"),
]

print(f"{'Date':<14} {'Tzolkin':<15} {'Haab':<15} {'Long Count':<15} Note")
print("-" * 75)
for y, m, d, note in dates:
    jdn = gregorian_to_jdn(y, m, d)
    tz = jdn_to_tzolkin(jdn)
    hb = jdn_to_haab(jdn)
    lc = jdn_to_long_count(jdn)
    tz_str = f"{tz[0]} {tz[1]}"
    hb_str = f"{hb[1]} {hb[2]}"
    lc_str = '.'.join(str(x) for x in reversed(lc))
    print(f"{y:04d}-{m:02d}-{d:02d}   {tz_str:<15} {hb_str:<15} {lc_str:<15} {note}")

In [ ]:
# Show all 20 Tzolk'in day names
print("The 20 Tzolk'in day names:")
for i, name in enumerate(TZOLKIN_DAY_NAMES):
    print(f"  {i:>2}: {name}")

print(f"\nThe 19 Haab' months (18 regular + Wayeb'):")
for i, name in enumerate(HAAB_MONTH_NAMES):
    suffix = " (5 unlucky days)" if name == "Wayeb'" else ""
    print(f"  {i:>2}: {name}{suffix}")

## 2. Encoding Configurations

The MayaCalendarEncoder supports multiple encoding strategies.

In [ ]:
dates = np.array(["2024-01-01", "2024-06-15", "2024-12-21"])

# Tzolk'in only, separate components
enc = MayaCalendarEncoder(
    components=["tzolkin"],
    tzolkin_encoding="separate",
    cyclical=False,
)
result = enc.fit_transform(dates)
print("Tzolk'in (separate, no cyclical):")
print(f"  Features: {list(enc.get_feature_names_out())}")
print(f"  Shape: {result.shape}")
print(f"  Values:\n{result}")

In [ ]:
# Cyclical encoding adds sin/cos pairs
enc_cyc = MayaCalendarEncoder(
    components=["tzolkin"],
    tzolkin_encoding="separate",
    cyclical=True,
)
result_cyc = enc_cyc.fit_transform(dates)
print("Tzolk'in (separate, with cyclical sin/cos):")
print(f"  Features: {list(enc_cyc.get_feature_names_out())}")
print(f"  Shape: {result_cyc.shape}")
print(f"  Range: [{result_cyc.min():.4f}, {result_cyc.max():.4f}]")

In [ ]:
# Full encoding with all components
enc_full = MayaCalendarEncoder(
    components=["tzolkin", "haab", "long_count"],
    cyclical=True,
    wayeb_flag=True,
    long_count_levels=3,
)
result_full = enc_full.fit_transform(dates)
print(f"Full encoding: {result_full.shape[1]} features")
for name in enc_full.get_feature_names_out():
    print(f"  {name}")

## 3. Cycle Visualization

Let's visualize how the different calendar cycles create unique patterns.

In [ ]:
from datetime import datetime, timedelta

# Generate a year of dates
start = datetime(2024, 1, 1)
year_dates = [(start + timedelta(days=i)).strftime("%Y-%m-%d") for i in range(365)]
year_array = np.array(year_dates)

# Get Tzolk'in positions
enc_tz = MayaCalendarEncoder(
    components=["tzolkin"],
    tzolkin_encoding="combined",
    cyclical=False, normalize=False
)
tz_vals = enc_tz.fit_transform(year_array)

print(f"Tzolk'in positions over 365 days:")
print(f"  Min: {tz_vals.min()}, Max: {tz_vals.max()}")
print(f"  Unique values: {len(np.unique(tz_vals))}")
print(f"  (260 unique in a 260-day cycle)")

# Count Wayeb' days
wayeb_count = sum(
    1 for d in year_dates
    if is_wayeb(gregorian_to_jdn(
        int(d[:4]), int(d[5:7]), int(d[8:10])
    ))
)
print(f"\nWayeb' days in 2024: {wayeb_count} (expected ~5)")

## 4. Practical Usage in ML Pipeline

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

# Synthetic time series with a 13-day cycle (Tzolk'in number)
np.random.seed(42)
n_days = 500
base = datetime(2020, 1, 1)
dates_train = np.array(
    [(base + timedelta(days=i)).strftime("%Y-%m-%d") for i in range(n_days)]
)

# Target has a 13-day periodic component
jdns = np.array([
    gregorian_to_jdn(int(d[:4]), int(d[5:7]), int(d[8:10]))
    for d in dates_train
])
y = np.sin(2 * np.pi * jdns / 13) * 5 + np.random.normal(0, 1, n_days)

# MCE should capture this 13-day cycle naturally
pipe = Pipeline([
    ("mce", MayaCalendarEncoder(
        components=["tzolkin"], cyclical=True
    )),
    ("rf", RandomForestRegressor(n_estimators=50, random_state=42)),
])

# Train/test split
split = int(n_days * 0.8)
pipe.fit(dates_train[:split], y[:split])
score = pipe.score(dates_train[split:], y[split:])
print(f"R² on test set (13-day cycle): {score:.4f}")
print("MCE naturally captures the Tzolk'in 13-number cycle!")